# Arithmetic-Init GPT-2 Ablation Pipeline
**Overnight autonomous run** — all experiments run top-to-bottom with idempotent resume.

Edit **Cell 1 (Config)** only. Then *Runtime → Run all*.

In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import os, time, sys

DRIVE_DIR     = '/content/drive/MyDrive/서강대학교/NLP_FINAL'
ARITH_INIT_PT = os.path.join(DRIVE_DIR, 'cot_large_integer_arithmetic_pretrain.pt')
TIME_BUDGET_H = 9.0
SEED          = 11711

NOTEBOOK_START = time.time()

# 학습 매트릭스: 커리큘럼 2x2 (curriculum x mixed-data). MA는 완료->skip.
EXPERIMENTS = [
    {'id': 'MA_plan_numaug',
     'init': 'arith', 'recipe': 'plan_numaug', 'rung': 'multiarith',
     'epochs': 40, 'lr': 1e-5, 'patience': 8, 'batch_size': 8},
    {'id': 'G_A1_direct',
     'init': 'arith', 'recipe': 'gsm8k', 'rung': 'gsm8k',
     'epochs': 12, 'lr': 1e-5, 'patience': 4, 'batch_size': 8, 'eval_n': 150},
    {'id': 'G_A2_mixed',
     'init': 'arith', 'recipe': 'gsm8k_plus_ma', 'rung': 'gsm8k',
     'epochs': 12, 'lr': 1e-5, 'patience': 4, 'batch_size': 8, 'eval_n': 150},
    {'id': 'G_B1_curriculum',
     'init': 'ckpt:MA_plan_numaug', 'recipe': 'gsm8k', 'rung': 'gsm8k',
     'epochs': 12, 'lr': 5e-6, 'patience': 4, 'batch_size': 8, 'eval_n': 150},
    {'id': 'G_B2_curriculum_mix',
     'init': 'ckpt:MA_plan_numaug', 'recipe': 'gsm8k_plus_ma', 'rung': 'gsm8k',
     'epochs': 12, 'lr': 5e-6, 'patience': 4, 'batch_size': 8, 'eval_n': 150},
]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('Experiments:', len(EXPERIMENTS), '| Budget(h):', TIME_BUDGET_H)


GPU: Tesla T4
Experiments: 5 | Budget(h): 9.0


In [3]:
import subprocess, sys, os

# Drive 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 작업 디렉토리 이동
os.chdir(DRIVE_DIR)
if DRIVE_DIR not in sys.path:
    sys.path.insert(0, DRIVE_DIR)
print('CWD:', os.getcwd())

# GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

# 의존성 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'einops', 'tqdm'], check=True)

os.makedirs('outputs', exist_ok=True)
print('Setup complete.')


Mounted at /content/drive
CWD: /content/drive/MyDrive/서강대학교/NLP_FINAL
GPU: Tesla T4
Setup complete.


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DATA GENERATION (idempotent)                      ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, subprocess

def _exists(p): return os.path.exists(p) and os.path.getsize(p) > 0

# 3-a: prepare base split (if not done)
if not _exists("data/multiarith_sft_train_base.txt"):
    print("Running prepare_multiarith.py ...")
    subprocess.run(["python", "prepare_multiarith.py"], check=True)
else:
    print("[skip] multiarith base split already exists")

# 3-b: number-substitution augmentation
if not _exists("data/multiarith_aug_problems.jsonl"):
    print("Running augment_multiarith.py --k 10 ...")
    subprocess.run(["python", "augment_multiarith.py", "--k", "10"], check=True)
else:
    print("[skip] number-aug problems already exist")

# 3-c: deterministic CoT for augmented problems
if not _exists("data/multiarith_aug_cot.jsonl"):
    print("Running generate_cot_det.py ...")
    subprocess.run(["python", "generate_cot_det.py"], check=True)
else:
    print("[skip] aug CoT already exists")

# 3-d: assemble number-aug SFT file
if not _exists("data/multiarith_sft_train_aug.txt"):
    print("Running assemble_multiarith.py ...")
    subprocess.run(["python", "assemble_multiarith.py"], check=True)
else:
    print("[skip] numaug SFT already assembled")

# 3-e: plan-then-solve CoT (base and aug)
if not _exists("data/multiarith_sft_train_plan.txt"):
    print("Running generate_cot_plan.py ...")
    subprocess.run(["python", "generate_cot_plan.py"], check=True)
else:
    print("[skip] plan CoT already exists")

# 3-f: entity augmentation
if not _exists("data/multiarith_entity_aug_problems.jsonl"):
    print("Running augment_entities.py --k 3 ...")
    subprocess.run(["python", "augment_entities.py", "--k", "3"], check=True)
else:
    print("[skip] entity-aug problems already exist")

# 3-g: CoT for entity-aug problems (reuse generate_cot_det.py with different input)
ENTITY_COT = "data/multiarith_entity_aug_cot.jsonl"
if not _exists(ENTITY_COT):
    print("Generating CoT for entity-aug problems ...")
    import ast, json, re
    sys.path.insert(0, os.getcwd())
    from generate_cot_det import make_cot
    from generate_cot_plan import make_plan_line, prepend_plan, cot_is_valid
    ok = err = 0
    with open("data/multiarith_entity_aug_problems.jsonl") as fin, \
         open(ENTITY_COT, "w") as fout:
        for line in fin:
            rec = json.loads(line)
            try:
                cot = make_cot(rec["equation"], rec["gold_answer"])
                plan = make_plan_line(rec["equation"])
                cot_with_plan = prepend_plan(cot, plan)
                row = dict(rec); row["cot"] = cot_with_plan
                fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                ok += 1
            except Exception as e:
                err += 1
    print(f"  entity CoT: {ok} ok, {err} err")
else:
    print("[skip] entity CoT already exists")

# 3-h: assemble entity-aug SFT file (plan + num-aug + entity-aug)
ENTITY_SFT = "data/multiarith_sft_train_entity_aug.txt"
if not _exists(ENTITY_SFT):
    print("Assembling entity-aug SFT file ...")
    import json, re
    _BLOCK = re.compile(r"<<\s*([0-9+\-*/().\s]+?)\s*=\s*(-?\d+(?:\.\d+)?)\s*>>")
    _FINAL = re.compile(r"####\s*(-?\d+(?:\.\d+)?)")
    def _valid(cot, gold):
        for expr, claimed in _BLOCK.findall(cot):
            try:
                if abs(eval(expr,{"__builtins__":{}},{}) - float(claimed)) > 1e-6: return False
            except: return False
        m = _FINAL.search(cot)
        return m and abs(float(m.group(1)) - float(gold)) <= 1e-6
    dev_ids = {json.loads(l)["id"] for l in open("data/multiarith_dev.jsonl")}
    blocks = []
    for src in ["data/multiarith_sft_train_plan_aug.txt"]:
        if _exists(src):
            blocks.append(open(src).read())
    with open(ENTITY_COT) as f:
        for line in f:
            rec = json.loads(line)
            if rec["src_id"] in dev_ids: continue
            if not _valid(rec["cot"], rec["gold_answer"]): continue
            ex_id = f"{rec['src_id']}_ent{rec['variant']}"
            blocks.append(f"{ex_id}\n\nQuestion: {rec['question'].strip()}\n\nReasoning:\n{rec['cot'].strip()}\n\n<|endoftext|>\n\n")
    with open(ENTITY_SFT, "w") as f:
        f.writelines(blocks)
    print(f"  entity_aug SFT: {len(blocks)} blocks -> {ENTITY_SFT}")
else:
    print("[skip] entity-aug SFT already assembled")

# 3-i: GSM8K + MultiArith combo
GSM8K_PLUS_MA     = "data/gsm8k_plus_ma_sft_train.txt"
GSM8K_PLUS_ENTITY = "data/gsm8k_plus_entity_sft_train.txt"
for (combo_path, ma_path) in [
    (GSM8K_PLUS_MA,     "data/multiarith_sft_train_plan_aug.txt"),
    (GSM8K_PLUS_ENTITY, ENTITY_SFT),
]:
    if not _exists(combo_path):
        print(f"Building {combo_path} ...")
        gsm_text = open("data/gsm8k_sft_train.txt").read()
        ma_text  = open(ma_path).read() if _exists(ma_path) else ""
        with open(combo_path, "w") as f:
            f.write(gsm_text)
            if ma_text: f.write("\n" + ma_text)
        print(f"  done")
    else:
        print(f"[skip] {combo_path} already exists")

# 3-j: template leakage audit
if not _exists("data/multiarith_dev_template_labels.jsonl"):
    print("Running audit_template_leakage.py ...")
    subprocess.run(["python", "audit_template_leakage.py",
                    "--train_sft", "data/multiarith_sft_train_plan_aug.txt"], check=True)
else:
    print("[skip] template labels already computed")

# ── recipe → path map (used by Cell 4) ───────────────────────────────────
RECIPE_PATH = {
    "base":            "data/multiarith_sft_train_base.txt",
    "numaug":          "data/multiarith_sft_train_aug.txt",
    "plan_base":       "data/multiarith_sft_train_plan.txt",
    "plan_numaug":     "data/multiarith_sft_train_plan_aug.txt",
    "entity_numaug":   ENTITY_SFT,
    "gsm8k":           "data/gsm8k_sft_train.txt",
    "gsm8k_plus_ma":   GSM8K_PLUS_MA,
    "gsm8k_plus_entity": GSM8K_PLUS_ENTITY,
}

print("\nRecipe → file check:")
for recipe, path in RECIPE_PATH.items():
    status = "OK" if _exists(path) else "MISSING"
    print(f"  {recipe:<25} {status}  {path}")

print("\nData generation complete.")

[skip] multiarith base split already exists
[skip] number-aug problems already exist
[skip] aug CoT already exists
[skip] numaug SFT already assembled
[skip] plan CoT already exists
[skip] entity-aug problems already exist
Generating CoT for entity-aug problems ...
  entity CoT: 0 ok, 1530 err
[skip] entity-aug SFT already assembled
[skip] data/gsm8k_plus_ma_sft_train.txt already exists
[skip] data/gsm8k_plus_entity_sft_train.txt already exists
[skip] template labels already computed

Recipe → file check:
  base                      OK  data/multiarith_sft_train_base.txt
  numaug                    OK  data/multiarith_sft_train_aug.txt
  plan_base                 OK  data/multiarith_sft_train_plan.txt
  plan_numaug               OK  data/multiarith_sft_train_plan_aug.txt
  entity_numaug             OK  data/multiarith_sft_train_entity_aug.txt
  gsm8k                     OK  data/gsm8k_sft_train.txt
  gsm8k_plus_ma             OK  data/gsm8k_plus_ma_sft_train.txt
  gsm8k_plus_entity    

In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — EXPERIMENT RUNNER                                 ║
# ╚══════════════════════════════════════════════════════════════╝
import gc, torch
import csv, json, os, time, traceback
import sys; sys.path.insert(0, os.getcwd())

from run_ablation import run_experiment

MASTER_CSV  = "outputs/results_master.csv"
RUN_LOG     = "outputs/run_log.txt"
os.makedirs("outputs", exist_ok=True)

CSV_FIELDS = [
    "exp_id", "init", "train_data", "rung",
    "best_epoch", "best_accuracy",
    "format_valid_rate", "no_answer_rate", "repetition_rate",
    "in_template_accuracy", "held_out_template_accuracy",
    "total_epochs_run", "elapsed_s",
]

def _append_csv(row: dict):
    write_header = not os.path.exists(MASTER_CSV)
    with open(MASTER_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        w.writerow(row)

def _log(msg: str):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(RUN_LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def _budget_ok() -> bool:
    elapsed_h = (time.time() - NOTEBOOK_START) / 3600
    return elapsed_h < TIME_BUDGET_H

all_results = []
_log(f"Starting {len(EXPERIMENTS)} experiments. Budget: {TIME_BUDGET_H}h")

for exp in EXPERIMENTS:
    exp_id = exp["id"]
    recipe = exp["recipe"]
    train_data = RECIPE_PATH.get(recipe)

    # Skip if recipe file is missing
    if not train_data or not os.path.exists(train_data):
        _log(f"[SKIP-missing] {exp_id}: recipe file '{recipe}' not found")
        continue

    # Check idempotent skip before time-budget check (already-done is free)
    metrics_path = f"outputs/{exp_id}_metrics.json"
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            r = json.load(f)
        all_results.append(r)
        _append_csv(r)
        _log(f"[ALREADY DONE] {exp_id}  acc={r.get('best_accuracy', '?'):.3f}")
        continue

    if not _budget_ok():
        _log(f"[BUDGET EXCEEDED] skipping remaining experiments")
        break

    cfg = {
        "id":         exp_id,
        "init":       exp["init"],
        "train_data": train_data,
        "rung":       exp["rung"],
        "epochs":     exp.get("epochs", 40),
        "lr":         exp.get("lr", 1e-5),
        "patience":   exp.get("patience", 8),
        "batch_size": exp.get("batch_size", 8),
        "eval_n":     exp.get("eval_n"),
    }

    try:
        result = run_experiment(
            cfg,
            arith_init_path="cot_large_integer_arithmetic_pretrain.pt",
            out_dir="outputs",
            seed=SEED,
            device_str="cuda",
        )
        all_results.append(result)
        _append_csv(result)
        _log(f"[DONE] {exp_id}  acc={result['best_accuracy']:.3f}  "
             f"elapsed={result['elapsed_s']:.0f}s")
    except Exception:
        tb = traceback.format_exc()
        _log(f"[ERROR] {exp_id}\n{tb}")

    # free GPU memory between experiments (prevents fragmentation OOM)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

_log(f"Experiment loop finished. {len(all_results)} results collected.")

[09:14:28] Starting 5 experiments. Budget: 9.0h
[09:14:30] [ALREADY DONE] MA_plan_numaug  acc=0.889

  EXP  : G_A1_direct
  init : arith  |  data : data/gsm8k_sft_train.txt
  rung : gsm8k  |  device : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [init] arith weights from cot_large_integer_arithmetic_pretrain.pt
  epoch   0  loss=2.6793  acc=0.0133  fmt=0.1400  elapsed=904s
  epoch   1  loss=1.9491  acc=0.0267  fmt=0.2800  elapsed=1745s
  epoch   2  loss=1.8134  acc=0.0200  fmt=0.4867  elapsed=2458s
  epoch   3  loss=1.7261  acc=0.0133  fmt=0.5667  elapsed=3108s
  epoch   4  loss=1.6619  acc=0.0200  fmt=0.4933  elapsed=3812s
  epoch   5  loss=1.6100  acc=0.0200  fmt=0.8000  elapsed=4322s
  [early stop] no improvement for 4 epochs.
  FINAL full-dev eval: acc=0.0240 (n=500)
  -> metrics saved to outputs/G_A1_direct_metrics.json
[11:02:10] [DONE] G_A1_direct  acc=0.024  elapsed=6404s

  EXP  : G_A2_mixed
  init : arith  |  data : data/gsm8k_plus_ma_sft_train.txt
  rung : gsm8k  |  device : cuda
Loaded examples: 6061
Examples missing EOS: 0
Min length: 70
Max length: 430
Average length: 133.04141230819997
First example preview:
"0\n\nQuestion: Tamara is 3 times Kim's height less 4 inches. Tamara and Kim have a combined height of 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [init] arith weights from cot_large_integer_arithmetic_pretrain.pt


: 

In [ ]:
# RESULTS TABLE + ANALYSIS (robust: legacy/eval-only 파일 무시)
import json, glob, os

exp_rows, eval_rows = [], []
for p in sorted(glob.glob('outputs/*_metrics.json')) + sorted(glob.glob('outputs/*_eval.json')):
    try:
        m = json.load(open(p))
    except Exception:
        continue
    if 'exp_id' in m:
        exp_rows.append(m)
    elif 'decoding' in m and m.get('rung'):
        m['_file'] = os.path.basename(p)
        eval_rows.append(m)

VANILLA_GSM8K = 0.016  # 기존 vanilla GPT-2 -> GSM8K SFT (참고 baseline)

def f3(x):
    return ('%.3f' % x) if isinstance(x, (int, float)) else 'N/A'

print('=== TRAINED EXPERIMENTS ===')
hdr = ('exp_id', 'init', 'rung', 'acc', 'in_tmpl', 'held_out', 'fmt', 'ep', 'sec')
print('{:<24}{:<14}{:<11}{:>7}{:>9}{:>10}{:>7}{:>5}{:>7}'.format(*hdr))
print('-' * 94)
for r in exp_rows:
    sec = str(int(r['elapsed_s'])) if 'elapsed_s' in r else '?'
    print('{:<24}{:<14}{:<11}{:>7}{:>9}{:>10}{:>7}{:>5}{:>7}'.format(
        str(r.get('exp_id', '?'))[:23], str(r.get('init', '?'))[:13], str(r.get('rung', '?')),
        f3(r.get('best_accuracy')), f3(r.get('in_template_accuracy')),
        f3(r.get('held_out_template_accuracy')), f3(r.get('format_valid_rate')),
        str(r.get('best_epoch', '?')), sec))

print('')
print('(reference) vanilla GPT-2 -> GSM8K SFT :', f3(VANILLA_GSM8K))

gsm = {r['exp_id']: r.get('best_accuracy') for r in exp_rows if r.get('rung') == 'gsm8k'}
if gsm:
    print('')
    print('=== GSM8K arms ===')
    for k in sorted(gsm):
        print('  {:<24} {}'.format(k, f3(gsm[k])))

if eval_rows:
    print('')
    print('=== EVAL-ONLY (zero-shot transfer / self-consistency) ===')
    for r in eval_rows:
        acc = r.get('exact_accuracy', r.get('self_consistency_accuracy'))
        extra = ''
        if 'any_correct_rate' in r:
            extra = '  pass@k=%s  agree=%.2f' % (f3(r['any_correct_rate']), r.get('mean_vote_agreement', 0))
        print('  {:<52} {:<22} acc={}{}'.format(r['_file'], str(r.get('decoding', '?')), f3(acc), extra))

print('')
print('Full rows: outputs/results_master.csv')


exp_id                              init     rung          acc  in_tmpl  held_out    fmt   ep    sec
----------------------------------------------------------------------------------------------------


KeyError: 'exp_id'

In [ ]:
# EVAL-ONLY (학습 X): zero-shot transfer + self-consistency
import os, sys, json, subprocess, glob

def _done(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

PY = sys.executable

MA_CKPT = 'outputs/MA_plan_numaug_best.pt'
ZS_OUT  = 'outputs/MA_plan_numaug_best_gsm8k_transfer_eval.json'
if _done(MA_CKPT) and not _done(ZS_OUT):
    print('>> zero-shot transfer: MA ckpt -> GSM8K dev')
    subprocess.run([PY, 'eval_checkpoint.py',
                    '--checkpoint', MA_CKPT, '--rung', 'gsm8k',
                    '--tag', 'transfer', '--use_gpu'], check=True)
else:
    print('[skip] zero-shot transfer')

SC_K = 8
sc_targets = []
gsm_metrics = []
for p in glob.glob('outputs/*_metrics.json'):
    try:
        m = json.load(open(p))
    except Exception:
        continue
    if m.get('rung') == 'gsm8k' and m.get('checkpoint') and _done(m['checkpoint']):
        gsm_metrics.append(m)
if gsm_metrics:
    best_gsm = max(gsm_metrics, key=lambda m: m.get('best_accuracy', 0))
    sc_targets.append((best_gsm['checkpoint'], 'gsm8k'))
if _done(MA_CKPT):
    sc_targets.append((MA_CKPT, 'multiarith'))

for ckpt, rung in sc_targets:
    base = os.path.splitext(os.path.basename(ckpt))[0]
    sc_out = 'outputs/' + base + '_' + rung + '_sc_k' + str(SC_K) + '_eval.json'
    if _done(sc_out):
        print('[skip] self-consistency', base, rung)
        continue
    print('>> self-consistency k=' + str(SC_K) + ':', base, rung)
    subprocess.run([PY, 'eval_self_consistency.py',
                    '--checkpoint', ckpt, '--rung', rung,
                    '--k', str(SC_K), '--use_gpu'], check=True)

print('Eval-only stage complete.')
